<a href="https://colab.research.google.com/github/lzxatdk-tech/NLP_project/blob/main/reliable_multilingual_question_answering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Environment Setup

## Mount data folder

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
folder_path = '/content/drive/MyDrive/Shared_NLP_Project'

os.chdir(folder_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Download or load dataset

In [4]:
import os
from datasets import load_dataset, load_from_disk

TARGET_DIR = "./tydi_xor_rc"

if not os.path.exists(TARGET_DIR):
    print(f"Downloading dataset to {TARGET_DIR}...")
    dataset = load_dataset("coastalcph/tydi_xor_rc")
    dataset.save_to_disk(TARGET_DIR)
    print("Download and save complete.")
else:
    print(f"Found existing dataset at {TARGET_DIR}. Loading from disk...")
    dataset = load_from_disk(TARGET_DIR)
    print("Loading completes.")

df_train = dataset["train"].to_pandas()
df_val = dataset["validation"].to_pandas()

Found existing dataset at ./tydi_xor_rc. Loading from disk...
Loading completes.


In [ ]:
# tokenizer
from transformers import AutoTokenizer

mbert_tokeniser = AutoTokenizer.from_pretrained("bert-base-multilingual-uncased")

## BIO Labels

In [ ]:
def bio_labelller(context, answer, answer_start, tokenizer):
    encoding = tokenizer(
        context,
        add_special_tokens=False,
        return_offsets_mapping=True,
    )

    tokens = tokenizer.convert_ids_to_tokens(encoding["input_ids"])
    offsets = encoding["offset_mapping"]

    labels = ["O"] * len(tokens)

    # Unanswerable example
    if not answer or answer_start is None or answer_start < 0:
        return tokens, offsets, labels

    answer_end = answer_start + len(answer)

    # Verify that the provided offset is correct
    if context[answer_start:answer_end] != answer:
        raise ValueError(
            f"Answer mismatch: expected {answer!r}, "
            f"found {context[answer_start:answer_end]!r}"
        )

    answer_token_indices = []

    for token_index, (token_start, token_end) in enumerate(offsets):
        # The token overlaps the answer's character interval
        overlaps_answer = (
            token_start < answer_end and
            token_end > answer_start
        )

        if overlaps_answer:
            answer_token_indices.append(token_index)

    if not answer_token_indices:
        raise ValueError("The answer does not overlap any token")

    labels[answer_token_indices[0]] = "B"

    for token_index in answer_token_indices[1:]:
        labels[token_index] = "I"

    return tokens, offsets, labels